# 🔮 Fase 5: Predicción de Vida Útil (LSTM)

Con los sensores seleccionados del C-MAPSS FD001, vamos a entrenar una **Red Neuronal Recurrente (LSTM)**. Las LSTMs tienen "memoria" y son ideales para predecir el RUL basándose en la secuencia histórica (ventana de tiempo) del comportamiento del motor.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import joblib

plt.style.use('dark_background')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

## 1. Carga de Datos y Preprocesamiento
A diferencia de modelos clásicos, la LSTM necesita datos en 3 dimensiones: `(muestras, tiempo, features)`.

In [ ]:
# Cargar datos (haremos el mismo cálculo de RUL del EDA rápidamente)
columns = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
train_df = pd.read_csv('../data/cmapss/train_FD001.txt', sep='\s+', header=None, names=columns)

max_cycles = train_df.groupby('unit_nr')['time_cycles'].max().reset_index()
max_cycles.rename(columns={'time_cycles': 'max_cycle'}, inplace=True)
train_df = train_df.merge(max_cycles, on=['unit_nr'], how='left')
train_df['RUL'] = (train_df['max_cycle'] - train_df['time_cycles']).clip(upper=125)

try:
    useful_sensors = joblib.load('../models/useful_sensors_cmapss.pkl')
except:
    # Fallback si no corriste el notebook 4
    useful_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

print(f"Usando {len(useful_sensors)} sensores.")

# Normalizamos con MinMaxScaler (ideal para redes neuronales)
scaler = MinMaxScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])
joblib.dump(scaler, '../models/scalers/lstm_scaler.pkl')

## 2. Ventanas Deslizantes (Sliding Windows)
Para cada instante $t$, tomamos los últimos 30 ciclos para predecir el RUL en el instante $t$.

In [ ]:
SEQUENCE_LENGTH = 30

def create_sequences(df, seq_length, cols):
    X, y = [], []
    for unit_id in df['unit_nr'].unique():
        unit_data = df[df['unit_nr'] == unit_id]
        data_matrix = unit_data[cols].values
        label_array = unit_data['RUL'].values
        
        for i in range(len(unit_data) - seq_length):
            X.append(data_matrix[i:i+seq_length])
            y.append(label_array[i+seq_length])
            
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df, SEQUENCE_LENGTH, useful_sensors)
print(f"Shape de X_train (muestras, tiempo, features): {X_train.shape}")
print(f"Shape de y_train: {y_train.shape}")

# Convertimos a Tensores
train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.FloatTensor(y_train).view(-1,1).to(device))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

## 3. Arquitectura LSTM

In [ ]:
class RULPredictorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(RULPredictorLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1) # Predice un único valor continuo (RUL)

    def forward(self, x):
        out, _ = self.lstm(x)
        # Tomamos solo el output del último paso de tiempo de la secuencia
        out = out[:, -1, :]
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out

model = RULPredictorLSTM(input_dim=len(useful_sensors)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 4. Entrenamiento

In [ ]:
epochs = 20
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss/len(train_loader)
    train_losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - MSE Loss: {avg_loss:.2f} - RMSE: {np.sqrt(avg_loss):.2f} ciclos")

plt.plot(train_losses, color='#e74c3c')
plt.title('Curva de Aprendizaje - LSTM')
plt.ylabel('MSE Loss')
plt.show()

In [ ]:
torch.save(model.state_dict(), '../models/lstm_rul.pt')
print("Modelo LSTM exportado exitosamente.")